In [1]:
import matplotlib.pyplot as plt

try:
    get_ipython().run_line_magic("matplotlib inline")
except:
    plt.ion()
import numpy as np

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import config as cfg
import torch
from utils.calibration_plots_2 import make_calibration_plots

C:\Users\MacRaeDC\AppData\Local\Temp\ipykernel_11704\1009679900.py:10: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was too old on your system - pyarrow 10.0.1 is the current minimum supported version as of this release.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
def preds_df_to_dict(all_val_preds):
    DL_pred_dict = dict()
    DL_true_dict = dict()
    for endpoint in cfg.endpoint_list:
        DL_pred_dict[endpoint] = torch.tensor(all_val_preds[endpoint+"_pred"].values)
        DL_true_dict[endpoint] = torch.tensor(all_val_preds[endpoint+"_true"].values)


    valid_endpoints_as_tensor = torch.as_tensor(cfg.valid_endpoint_values)

    for endpoint in cfg.endpoint_list:
        mask = torch.isin(DL_true_dict[endpoint], valid_endpoints_as_tensor)

        DL_pred_dict[endpoint] = DL_pred_dict[endpoint][mask]
        DL_true_dict[endpoint] = DL_true_dict[endpoint][mask]

    return DL_pred_dict, DL_true_dict



In [3]:
experiment_name = "HP_ViT"
exp_path = os.path.join(os.getcwd(), "HP_experiments", "DCNN", experiment_name)
exp_path = os.path.join(os.getcwd(), "HP_experiments", experiment_name)
#exp_path = os.path.join(os.getcwd(), "HP_experiments", "TransferLearning", experiment_name)


df_results = pd.read_csv(os.path.join(exp_path, "recovered_results.csv"), delimiter=";")
df_results.sort_values(by="mean_val_AUC", ascending=False, inplace=True)
df_results = df_results.head(3)
df_results
top_3_IDs = list(df_results["trial_id"].values)

model_name = "ViT"

In [4]:
top_3_IDs

['c7dbdcd7', 'b36e8dd6', 'bfc5ca0f']

In [6]:
for trial_id in top_3_IDs:
    all_val_preds = []
    # get the 5 folders for the 5 folds
    trial_path = os.path.join(exp_path, str(trial_id))
    fold_folders = glob.glob(trial_path + "/*/", recursive = True)
    fold_folders = [folder for folder in fold_folders if "checkpoint" not in folder]
    
    
    for fold_folder in fold_folders:
        # find the model predictions file
        fold_folder_path = os.path.join(trial_path, fold_folder)
        all_files = os.listdir(fold_folder_path)
        print(all_files)
        prediction_csv = [file for file in all_files if (("all_outputs" in file) and (model_name in file))][0]
        predictions_path = os.path.join(fold_folder_path, prediction_csv)
        
        # load the predictions and extract only the validation predictions
        df_fold_preds = pd.read_csv(predictions_path, delimiter=";")
        fold_val_preds = df_fold_preds[df_fold_preds["Mode"] != "val"]
        all_val_preds.append(fold_val_preds.copy())
    
    all_val_preds = pd.concat(all_val_preds)

    DL_pred_dict, DL_true_dict = preds_df_to_dict(all_val_preds)

    filename = f"{trial_id}_all_folds_validation_prediction.png"
    save_plot_path = os.path.join(trial_path, filename)

    _, _, _, _ = make_calibration_plots(cfg, DL_pred_dict, DL_true_dict, None, None, set_name=f"all validation - trial {trial_id}", mode = "calibration", filename=save_plot_path)

    #filename = f"{trial_id}_all_folds_validation_prediction.png"
    save_plot_path = os.path.join(os.getcwd(), "HP_experiments", "Calibration Plots", experiment_name, filename)
    print(save_plot_path)
    _, _, _, _ = make_calibration_plots(cfg, DL_pred_dict, DL_true_dict, None, None, set_name=f"all validation - trial {trial_id}", mode = "calibration", filename=save_plot_path)


['all_toxicities losses.png', 'calibration_plot_validation.png', 'figures', 'log.txt', 'lr_all_outputs.csv', 'model.txt', 'reliability_plot_validation.png', 'results.csv', 'results.png', 'Thumbs.db', 'train_val_test_patient_ids.json', 'ViT_all_outputs.csv']
['all_toxicities losses.png', 'calibration_plot_validation.png', 'figures', 'log.txt', 'lr_all_outputs.csv', 'model.txt', 'reliability_plot_validation.png', 'results.csv', 'results.png', 'Thumbs.db', 'train_val_test_patient_ids.json', 'ViT_all_outputs.csv']
['all_toxicities losses.png', 'calibration_plot_validation.png', 'figures', 'log.txt', 'lr_all_outputs.csv', 'model.txt', 'reliability_plot_validation.png', 'results.csv', 'results.png', 'Thumbs.db', 'train_val_test_patient_ids.json', 'ViT_all_outputs.csv']
['all_toxicities losses.png', 'calibration_plot_validation.png', 'figures', 'log.txt', 'lr_all_outputs.csv', 'model.txt', 'reliability_plot_validation.png', 'results.csv', 'results.png', 'Thumbs.db', 'train_val_test_patient_id